# Train the polyp segmentation model on Colab

**Before you start**

1. Runtime > Change runtime type > **T4 GPU**.
2. Put these in the top level of your Google Drive ("My Drive"):
   - `archive.zip`: the Kvasir-SEG download (nested folders are fine)
   - the project, either as `polyp-segmentation.zip` or as the uploaded `polyp-segmentation` folder

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: switch the runtime type to GPU first"
print(torch.cuda.get_device_name(0))

## 1. Mount Google Drive

Results are written to Drive, so the checkpoint survives if Colab disconnects.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/polypseg_runs/unet_resnet34"

## 2. Unpack the code and the dataset

The project is taken from `polyp-segmentation.zip` if it is in Drive, otherwise from the
`polyp-segmentation` folder. Edit `DRIVE` if your files are not at the top level.

In [ ]:
import glob
import os
import shutil
import sys
import zipfile

DRIVE = "/content/drive/MyDrive"
WORK = "/content"

code_dir = f"{WORK}/code"
os.makedirs(code_dir, exist_ok=True)
if os.path.exists(f"{DRIVE}/polyp-segmentation.zip"):
    zipfile.ZipFile(f"{DRIVE}/polyp-segmentation.zip").extractall(code_dir)
else:
    shutil.copytree(
        f"{DRIVE}/polyp-segmentation",
        f"{code_dir}/polyp-segmentation",
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns(".venv", "data", "runs", "__pycache__"),
    )
PROJECT = os.path.dirname(min(glob.glob(f"{code_dir}/**/pyproject.toml", recursive=True), key=len))
print("Project folder:", PROJECT)

zipfile.ZipFile(f"{DRIVE}/archive.zip").extractall(f"{WORK}/data")
print("Dataset unpacked to", f"{WORK}/data")

In [ ]:
%cd {PROJECT}
!pip install -q -e ".[train]"

In [ ]:
# Sanity check before training: the dataset must be found (Kvasir-SEG has 1000 pairs)
sys.path.insert(0, f"{PROJECT}/src")
from polypseg.data import list_pairs

print(len(list_pairs(f"{WORK}/data")), "image/mask pairs found (expect 1000)")

*Alternatives (not needed for the first run).* With the code on GitHub, replace the project part of
step 2 by `!git clone https://github.com/<your-username>/polyp-segmentation.git /content/code/polyp-segmentation`.
To download the dataset instead of using your own copy: `!python -m polypseg.download --out-dir /content/data`.

## 3. Train

Defaults come from `configs/default.yaml` (50 epochs). For a quick trial add
`--override train.epochs=3` to the command.

In [ ]:
!python -m polypseg.train --data-dir /content/data --out-dir {OUT_DIR} --config configs/default.yaml

## 4. Inspect the results

In [ ]:
import json

from IPython.display import Image, display

metrics = json.load(open(f"{OUT_DIR}/metrics.json"))
keys = ("best_epoch", "validation", "test", "test_dice_median", "test_dice_std",
        "parameters", "train_minutes", "device")
print(json.dumps({k: metrics[k] for k in keys}, indent=2))
for name in ("curves.png", "samples.png", "failure_cases.png"):
    display(Image(f"{OUT_DIR}/{name}"))

## 5. Get the files back

Everything is already in your Drive under `polypseg_runs/unet_resnet34/`. This cell also downloads it
as a zip. Put `best.pt` in the project's `models/` folder and copy the three PNGs into `docs/`.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/polypseg_results", "zip", OUT_DIR)
files.download("/content/polypseg_results.zip")